<a href="https://colab.research.google.com/github/AngelTroncoso/Alergias/blob/main/03_VerAlergenos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vista de Moleculas para comparar propiedades:

In [3]:
pip install py3Dmol

In [4]:
import py3Dmol

# Create a 3Dmol viewer object
view = py3Dmol.view(query='pdb:1AON') # You can replace '1AON' with any PDB ID

# Set the style to cartoon and add a surface
view.setStyle({'cartoon': {'color': 'spectrum'}})
view.addSurface(py3Dmol.VDW, {'opacity':0.5})

# Display the viewer
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Alergenos a Analizar

# Der p 1, Der f 1, Der p 2, Der p 7 y Der p 23, respectivamente: ['1GP2', '1DF1', '1POA', '1K6U', '5DM2']



In [8]:
import py3Dmol

# Esta lista contendrá los IDs de PDB de los alérgenos que identificaste manualmente.
# Los siguientes IDs son para Der p 1, Der f 1, Der p 2, Der p 7 y Der p 23, respectivamente.
allergen_pdb_ids = ['1GP2', '1DF1', '1POA', '1K6U', '5DM2']

print("Visualizando los alérgenos:\n")

for pdb_id in allergen_pdb_ids:
    print(f"--- PDB ID: {pdb_id} ---")
    view = py3Dmol.view(query=f'pdb:{pdb_id}')
    view.setStyle({'cartoon': {'color': 'spectrum'}})
    view.addSurface(py3Dmol.VDW, {'opacity': 0.5})
    view.show()
    print("\n")

Visualizando los alérgenos:

--- PDB ID: 1GP2 ---


3Dmol.js failed to load for some reason. Please check your browser console for error messages.



--- PDB ID: 1DF1 ---


3Dmol.js failed to load for some reason. Please check your browser console for error messages.



--- PDB ID: 1POA ---


3Dmol.js failed to load for some reason. Please check your browser console for error messages.



--- PDB ID: 1K6U ---


3Dmol.js failed to load for some reason. Please check your browser console for error messages.



--- PDB ID: 5DM2 ---


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Este ejemplo carga la entrada PDB '1AON' (una molécula de proteína) y la muestra con un estilo de caricatura y una superficie semi-transparente. Puedes reemplazar `pdb:1AON` con otros ID de PDB o incluso cargar datos desde una cadena usando `view.addModel(data, 'pdb')`.

### Comparación Estructural por Superposición

Podemos usar `py3Dmol` para superponer las estructuras de los alérgenos. Esto nos permite comparar visualmente sus formas e identificar características estructurales comunes o diferencias.

In [7]:
import py3Dmol

# Asegúrese de que la lista de IDs de PDB de alérgenos esté disponible
# Si ejecutas esta celda de forma independiente, descomenta la línea a continuación:
# allergen_pdb_ids = ['1GP2', '1DF1', '1POA', '1K6U', '5DM2']

if 'allergen_pdb_ids' not in locals() and 'allergen_pdb_ids' not in globals():
    print("Error: 'allergen_pdb_ids' not defined. Please run the cell defining the list of PDB IDs first.")
else:
    view_superimposed = py3Dmol.view(query='pdb:' + allergen_pdb_ids[0])
    view_superimposed.setStyle({'cartoon': {'color': 'spectrum'}})

    # Cargar y superponer el resto de las moléculas
    for i, pdb_id in enumerate(allergen_pdb_ids[1:]):
        # Corregido: Proporcione la URL del archivo PDB directamente
        view_superimposed.addModel(f"https://files.rcsb.org/download/{pdb_id}.pdb", 'pdb')
        view_superimposed.setStyle({'model': i+1}, {'cartoon': {'color': 'orange'}})
        view_superimposed.superposition(viewer=(i+1), target=0)

    # Opcionalmente, restablezca la vista y agregue una superficie
    view_superimposed.zoomTo()
    view_superimposed.addSurface(py3Dmol.VDW, {'opacity':0.3, 'color': 'lightgray'})

    print("--- Superimposed Allergen Structures ---")
    view_superimposed.show()

--- Superimposed Allergen Structures ---


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

En la vista superpuesta, el primer alérgeno (`1GP2`) se muestra en un espectro de colores, mientras que los demás se muestran en naranja. Esto ayuda a distinguirlos y observar qué tan estrechamente se alinean sus estructuras.

# Alineación de Secuencias para Comparar Pares de Alérgenos

### Visualización de Regiones de Alta Identidad en 3D

In [22]:
from Bio.PDB import PDBList, PDBParser
from Bio.PDB.Polypeptide import protein_letters_3to1 # Correct import for AA conversion
from Bio import SeqIO
from Bio import pairwise2
import py3Dmol
import os

# Asegurémonos de que all_sequences esté cargado
if 'all_sequences' not in locals() and 'all_sequences' not in globals():
    print("Error: 'all_sequences' no está definido. Por favor, ejecuta las celdas de alineación de secuencias primero.")
else:
    # Seleccionar el nuevo par para visualizar (1GP2:A y 1POA:A)
    ref_seq_record = next((seq for seq in all_sequences if seq.id.startswith('1GP2')), None)
    target_seq_record = next((seq for seq in all_sequences if seq.id.startswith('1POA')), None) # Cambiado a 1POA

    if not ref_seq_record or not target_seq_record:
        print("No se pudieron encontrar los registros de secuencia para 1GP2 o 1POA.")
    else:
        print(f"Preparando visualización para {ref_seq_record.id} y {target_seq_record.id}")

        # Realizar la alineación nuevamente para este par
        alignments = pairwise2.align.globalms(
            ref_seq_record.seq, target_seq_record.seq, 2, -1, -0.5, -0.1
        )

        if alignments:
            best_alignment = alignments[0]
            seq1_aligned, seq2_aligned, score, begin, end = best_alignment

            print(f"Alineación entre {ref_seq_record.id} y {target_seq_record.id} (puntuación: {score:.2f})")

            # --- Identificar regiones de alta identidad (segmentos idénticos) ---
            identical_regions_ref = [] # Contendrá tuplas (start_idx_orig, end_idx_orig)
            identical_regions_target = []

            current_identical_segment_ref_start = -1
            current_identical_segment_target_start = -1

            # Track original sequence indices for each aligned position
            orig_idx_ref = 0
            orig_idx_target = 0

            for i in range(len(seq1_aligned)):
                char1 = seq1_aligned[i]
                char2 = seq2_aligned[i]

                is_match = (char1 == char2 and char1 != '-')

                if is_match:
                    if current_identical_segment_ref_start == -1: # Start of a new identical segment
                        current_identical_segment_ref_start = orig_idx_ref
                        current_identical_segment_target_start = orig_idx_target
                else:
                    if current_identical_segment_ref_start != -1: # End of an identical segment
                        identical_regions_ref.append((current_identical_segment_ref_start, orig_idx_ref - 1))
                        identical_regions_target.append((current_identical_segment_target_start, orig_idx_target - 1))
                        current_identical_segment_ref_start = -1
                        current_identical_segment_target_start = -1

                if char1 != '-': # Advance original sequence index if not a gap
                    orig_idx_ref += 1
                if char2 != '-':
                    orig_idx_target += 1

            # Add any trailing identical segment
            if current_identical_segment_ref_start != -1:
                identical_regions_ref.append((current_identical_segment_ref_start, orig_idx_ref - 1))
                identical_regions_target.append((current_identical_segment_target_start, orig_idx_target - 1))

            print("\nRegiones idénticas en la secuencia original (0-indexado):")
            print(f"  {ref_seq_record.id}: {identical_regions_ref}")
            print(f"  {target_seq_record.id}: {identical_regions_target}")

            # --- Cargar estructuras PDB y mapear residuos ---
            parser = PDBParser()
            pdb_list = PDBList()

            # Function to get residue ranges from sequence indices, with robust mapping
            def get_pdb_residue_ranges(pdb_id, sequence_regions_from_seqio, seq_record_from_seqio):
                pdb_file = pdb_list.retrieve_pdb_file(pdb_id, pdir='.', file_format='pdb')
                structure = parser.get_structure(pdb_id, pdb_file)

                mapped_residue_ranges = []

                for model in structure:
                    for chain in model:
                        pdb_atom_sequence = [] # List of 1-letter AA codes from ATOM records
                        pdb_res_id_map = {}   # Map 0-indexed PDB_atom_sequence position to PDB res_id (tuple)

                        # Extract sequence and PDB res_ids from ATOM records for the current chain
                        for i, res in enumerate(chain):
                            # Only consider standard amino acids (no HETATM, no water)
                            if res.has_id('CA') and res.get_resname() in protein_letters_3to1: # Use protein_letters_3to1
                                try:
                                    one_letter_code = protein_letters_3to1[res.get_resname()] # Use protein_letters_3to1
                                    pdb_atom_sequence.append(one_letter_code)
                                    # Map 0-indexed position in pdb_atom_sequence to PDB res_id (number)
                                    pdb_res_id_map[len(pdb_atom_sequence) - 1] = res.get_id()[1]
                                except KeyError:
                                    pass # Skip non-standard amino acids or water

                        # Convert list of 1-letter codes to a single string sequence
                        pdb_atom_sequence_str = "".join(pdb_atom_sequence)

                        # Align the SeqIO sequence with the PDB ATOM sequence to create a mapping
                        if pdb_atom_sequence_str and str(seq_record_from_seqio.seq):
                            alignments_mapping = pairwise2.align.globalms(
                                str(seq_record_from_seqio.seq), pdb_atom_sequence_str, 2, -1, -0.5, -0.1
                            )
                            if alignments_mapping:
                                best_alignment_map = alignments_mapping[0]
                                seqio_aligned_map, pdb_aligned_map, _, _, _ = best_alignment_map

                                # Create a map from SeqIO original index to PDB ATOM original index (which then maps to PDB res_id)
                                seqio_to_pdb_atom_idx_map = {}
                                seqio_orig_idx_counter = 0
                                pdb_atom_orig_idx_counter = 0

                                for s_char, p_char in zip(seqio_aligned_map, pdb_aligned_map):
                                    if s_char != '-' and p_char != '-': # If both are not gaps, they align
                                        seqio_to_pdb_atom_idx_map[seqio_orig_idx_counter] = pdb_atom_orig_idx_counter

                                    if s_char != '-':
                                        seqio_orig_idx_counter += 1
                                    if p_char != '-':
                                        pdb_atom_orig_idx_counter += 1

                                # Now, translate the identical_regions (which are SeqIO indices)
                                # to PDB residue ranges using the map
                                for start_orig_idx, end_orig_idx in sequence_regions_from_seqio:
                                    # Check if both start and end indices can be mapped
                                    if start_orig_idx in seqio_to_pdb_atom_idx_map and end_orig_idx in seqio_to_pdb_atom_idx_map:
                                        pdb_atom_start_idx = seqio_to_pdb_atom_idx_map[start_orig_idx]
                                        pdb_atom_end_idx = seqio_to_pdb_atom_idx_map[end_orig_idx]

                                        if pdb_atom_start_idx in pdb_res_id_map and pdb_atom_end_idx in pdb_res_id_map:
                                            res_num_start = pdb_res_id_map[pdb_atom_start_idx]
                                            res_num_end = pdb_res_id_map[pdb_atom_end_idx]

                                            # Ensure order is correct, though alignment should handle this for regions
                                            if res_num_start <= res_num_end:
                                                mapped_residue_ranges.append({'chain': chain.id, 'start': res_num_start, 'end': res_num_end})
                                            else:
                                                # Handle reversed ranges if they occur (e.g., in circular permutations, though unlikely here)
                                                mapped_residue_ranges.append({'chain': chain.id, 'start': res_num_end, 'end': res_num_start})
                                        # else:
                                        #     print(f"Warning: PDB ATOM original index for region [{start_orig_idx}, {end_orig_idx}] for {pdb_id} chain {chain.id} not found in PDB res_id map.")
                                    # else:
                                    #     print(f"Warning: SeqIO original index for region [{start_orig_idx}, {end_orig_idx}] for {pdb_id} chain {chain.id} not found in SeqIO to PDB ATOM index map.")
                        # else:
                        #     print(f"Warning: No alignable ATOM sequence found for {pdb_id} chain {chain.id}.")

                # Clean up the downloaded PDB file
                if os.path.exists(pdb_file):
                    os.remove(pdb_file)
                return mapped_residue_ranges

            # Get residue ranges for both proteins
            pdb_ranges_ref = get_pdb_residue_ranges(ref_seq_record.id[:4], identical_regions_ref, ref_seq_record)
            pdb_ranges_target = get_pdb_residue_ranges(target_seq_record.id[:4], identical_regions_target, target_seq_record)

            print(f"\nRangos de residuos PDB para {ref_seq_record.id}: {pdb_ranges_ref}")
            print(f"Rangos de residuos PDB para {target_seq_record.id}: {pdb_ranges_target}")

            # --- Visualización 3D con py3Dmol ---
            view = py3Dmol.view(query=f'pdb:{ref_seq_record.id[:4]}') # Cargar el primero
            view.setStyle({'cartoon': {'color': 'lightgray'}})

            # Colorear las regiones idénticas en el alérgeno de referencia (e.g., rojo)
            for res_range in pdb_ranges_ref:
                view.addStyle({
                    'chain': res_range['chain'],
                    'resi': [res_range['start'], res_range['end']],
                    'and': {'atom': 'CA'} # Target C-alpha atoms in the range
                }, {'cartoon': {'color': 'red'}})

            # Añadir el segundo alérgeno y colorear sus regiones idénticas (e.g., azul)
            view.addModel(f"https://files.rcsb.org/download/{target_seq_record.id[:4]}.pdb", 'pdb')
            view.setStyle({'model': 1, 'cartoon': {'color': 'lightgreen'}})
            for res_range in pdb_ranges_target:
                view.addStyle({
                    'model': 1,
                    'chain': res_range['chain'],
                    'resi': [res_range['start'], res_range['end']],
                    'and': {'atom': 'CA'}
                }, {'cartoon': {'color': 'blue'}})

            view.superposition(viewer=1, target=0) # Superponer
            view.zoomTo()
            view.addSurface(py3Dmol.VDW, {'opacity':0.3, 'color': 'lightgray'})

            print("--- Visualización de regiones alineadas de alta identidad ---")
            view.show()

            print("\n--- Interpretación de la Visualización ---")
            print("Las regiones coloreadas en rojo (para 1GP2:A) y azul (para 1POA:A) representan los segmentos de la proteína que son idénticos y sin 'gaps' en la alineación de secuencias. Su superposición en 3D nos permite observar si estas regiones conservadas también se superponen espacialmente, lo que podría indicar una importancia estructural o funcional compartida. Si el mapeo de residuos parece incorrecto (por ejemplo, si las regiones coloreadas no tienen sentido en la estructura), es posible que necesitemos un método de mapeo de secuencias a PDB más robusto.")

        else:
            print(f"No se encontró alineación entre {ref_seq_record.id} y {target_seq_record.id}.")

Preparando visualización para 1GP2:A y 1POA:A
Alineación entre 1GP2:A y 1POA:A (puntuación: 66.10)

Regiones idénticas en la secuencia original (0-indexado):
  1GP2:A: [(20, 21), (59, 59), (66, 66), (68, 68), (74, 74), (76, 77), (78, 78), (84, 84), (92, 93), (99, 100), (103, 103), (110, 110), (115, 115), (132, 133), (134, 134), (148, 148), (156, 159), (170, 170), (172, 172), (186, 186), (191, 191), (212, 212), (214, 214), (233, 234), (246, 246), (251, 251), (252, 252), (256, 256), (257, 257), (260, 260), (261, 261), (273, 273), (279, 279), (281, 282), (284, 284), (290, 290), (292, 292), (297, 299), (303, 303), (307, 307), (311, 311), (317, 317), (323, 323), (324, 324), (326, 326), (329, 329), (335, 335), (344, 344), (345, 347), (349, 349)]
  1POA:A: [(0, 1), (2, 2), (3, 3), (5, 5), (6, 6), (8, 9), (14, 14), (15, 15), (19, 20), (21, 22), (29, 29), (30, 30), (31, 31), (32, 33), (36, 36), (37, 37), (38, 41), (44, 44), (45, 45), (46, 46), (47, 47), (49, 49), (52, 52), (53, 54), (55, 55), (

/usr/local/lib/python3.12/dist-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 6338.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning: WARNING: Chain B is discontinuous at line 6591.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning: WARNING: Chain G is discontinuous at line 6733.
  warnings.warn(



Rangos de residuos PDB para 1GP2:A: [{'chain': 'A', 'start': 22, 'end': 23}, {'chain': 'A', 'start': 61, 'end': 61}, {'chain': 'A', 'start': 68, 'end': 68}, {'chain': 'A', 'start': 70, 'end': 70}, {'chain': 'A', 'start': 76, 'end': 76}, {'chain': 'A', 'start': 78, 'end': 79}, {'chain': 'A', 'start': 80, 'end': 80}, {'chain': 'A', 'start': 86, 'end': 86}, {'chain': 'A', 'start': 94, 'end': 95}, {'chain': 'A', 'start': 101, 'end': 102}, {'chain': 'A', 'start': 105, 'end': 105}, {'chain': 'A', 'start': 112, 'end': 112}, {'chain': 'A', 'start': 117, 'end': 117}, {'chain': 'A', 'start': 134, 'end': 135}, {'chain': 'A', 'start': 136, 'end': 136}, {'chain': 'A', 'start': 150, 'end': 150}, {'chain': 'A', 'start': 158, 'end': 161}, {'chain': 'A', 'start': 172, 'end': 172}, {'chain': 'A', 'start': 174, 'end': 174}, {'chain': 'A', 'start': 188, 'end': 188}, {'chain': 'A', 'start': 193, 'end': 193}, {'chain': 'A', 'start': 214, 'end': 214}, {'chain': 'A', 'start': 216, 'end': 216}, {'chain': 'A',

3Dmol.js failed to load for some reason. Please check your browser console for error messages.


--- Interpretación de la Visualización ---
Las regiones coloreadas en rojo (para 1GP2:A) y azul (para 1POA:A) representan los segmentos de la proteína que son idénticos y sin 'gaps' en la alineación de secuencias. Su superposición en 3D nos permite observar si estas regiones conservadas también se superponen espacialmente, lo que podría indicar una importancia estructural o funcional compartida. Si el mapeo de residuos parece incorrecto (por ejemplo, si las regiones coloreadas no tienen sentido en la estructura), es posible que necesitemos un método de mapeo de secuencias a PDB más robusto.


In [18]:
pip install biopython

In [19]:
from Bio.PDB import PDBList
from Bio import SeqIO
from Bio import pairwise2
from Bio.pairwise2 import format_alignment

# Helper function to calculate sequence identity (percentage) using the shorter original sequence length
def calculate_identity(seq1_orig, seq2_orig, seq1_aligned, seq2_aligned):
    matches = 0
    # Count actual matches between non-gap characters
    for s1_char, s2_char in zip(seq1_aligned, seq2_aligned):
        if s1_char == s2_char and s1_char != '-':
            matches += 1

    # Use the length of the shorter original sequence for the denominator
    shorter_orig_len = min(len(seq1_orig), len(seq2_orig))

    if shorter_orig_len == 0:
        return 0.0
    return (matches / shorter_orig_len) * 100

In [20]:
# Asegúrese de que la lista de IDs de PDB de alérgenos esté disponible
# Si ejecutas esta celda de forma independiente, descomenta la línea a continuación:
# allergen_pdb_ids = ['1GP2', '1DF1', '1POA', '1K6U', '5DM2']

if 'allergen_pdb_ids' not in locals() and 'allergen_pdb_ids' not in globals():
    print("Error: 'allergen_pdb_ids' not defined. Please run the cell defining the list of PDB IDs first.")
else:
    pdb_list = PDBList()
    all_sequences = []

    print("Obteniendo secuencias de proteínas de los IDs de PDB:")
    for pdb_id in allergen_pdb_ids:
        print(f"  Buscando secuencia para {pdb_id}...")
        try:
            # Descargar el archivo PDB en el directorio actual
            pdb_file_path = pdb_list.retrieve_pdb_file(pdb_id, pdir='.', file_format='pdb')

            # Intentar parsear la secuencia de los registros SEQRES en el archivo PDB
            # Tomar la primera cadena de proteína encontrada
            seq_found = False
            for record in SeqIO.parse(pdb_file_path, "pdb-seqres"):
                if record.seq and 'X' not in record.seq: # Basic check for valid protein sequence
                    all_sequences.append(record)
                    print(f"    Secuencia encontrada para {record.id} (longitud: {len(record.seq)}) ")
                    seq_found = True
                    break # Tomar solo la primera cadena significativa
            if not seq_found:
                print(f"    No se pudo extraer una secuencia de proteína válida de {pdb_id} de los registros SEQRES.")

            # Opcional: limpiar el archivo PDB descargado
            import os
            if os.path.exists(pdb_file_path):
                os.remove(pdb_file_path)

        except Exception as e:
            print(f"  Error al procesar {pdb_id}: {e}")

    if not all_sequences:
        print("No se pudieron recuperar secuencias para la alineación.")
    else:
        print("\nSecuencias recuperadas:")
        for seq_rec in all_sequences:
            print(f"  ID: {seq_rec.id}, Longitud: {len(seq_rec.seq)}")

Obteniendo secuencias de proteínas de los IDs de PDB:
  Buscando secuencia para 1GP2...
    Secuencia encontrada para 1GP2:A (longitud: 353) 
  Buscando secuencia para 1DF1...
    Secuencia encontrada para 1DF1:A (longitud: 423) 
  Buscando secuencia para 1POA...
    Secuencia encontrada para 1POA:A (longitud: 118) 
  Buscando secuencia para 1K6U...
    Secuencia encontrada para 1K6U:A (longitud: 58) 
  Buscando secuencia para 5DM2...
    Secuencia encontrada para 5DM2:A (longitud: 265) 

Secuencias recuperadas:
  ID: 1GP2:A, Longitud: 353
  ID: 1DF1:A, Longitud: 423
  ID: 1POA:A, Longitud: 118
  ID: 1K6U:A, Longitud: 58
  ID: 5DM2:A, Longitud: 265


In [21]:
if len(all_sequences) < 2:
    print("Se necesitan al menos dos secuencias para realizar una alineación por pares.")
else:
    print("\nRealizando alineaciones de secuencias por pares (usando el primer alérgeno como referencia):\n")

    ref_seq_record = all_sequences[0]
    ref_id = ref_seq_record.id
    print(f"Alérgeno de referencia: {ref_id}")

    for i in range(1, len(all_sequences)):
        current_seq_record = all_sequences[i]
        current_id = current_seq_record.id

        print(f"\n--- Comparando {ref_id} con {current_id} ---")
        # Realizar alineación global usando Bio.pairwise2.align.globalms
        # match=2, mismatch=-1, gap_open=-0.5, gap_extend=-0.1 son valores comunes
        alignments = pairwise2.align.globalms(ref_seq_record.seq, current_seq_record.seq, 2, -1, -0.5, -0.1)

        if alignments:
            best_alignment = alignments[0]
            score = best_alignment.score
            seq1_aligned, seq2_aligned, _, _, _ = best_alignment

            # Call the modified calculate_identity function with original sequences
            identity_percentage = calculate_identity(ref_seq_record.seq, current_seq_record.seq, seq1_aligned, seq2_aligned)

            print(f"  Puntuación de alineación: {score:.2f}")
            print(f"  Identidad de secuencia (vs. secuencia original más corta): {identity_percentage:.2f}%")
            # Opcionalmente, imprimir la alineación completa (puede ser muy larga)
            # print(format_alignment(*best_alignment))
        else:
            print(f"  No se encontró alineación entre {ref_id} y {current_id}.")

    print("\n--- Interpretación --- ")
    print("Una mayor puntuación de alineación y un mayor porcentaje de identidad de secuencia indican una mayor similitud entre las proteínas. Las regiones con alta similitud pueden corresponder a dominios estructurales o funcionales conservados, incluyendo potencialmente los sitios activos. Sin embargo, para identificar con precisión los sitios activos y sus similitudes, se necesitaría un análisis más detallado que incorpore información estructural o de anotación funcional específica de cada alérgeno.")


Realizando alineaciones de secuencias por pares (usando el primer alérgeno como referencia):

Alérgeno de referencia: 1GP2:A

--- Comparando 1GP2:A con 1DF1:A ---
  Puntuación de alineación: 156.80
  Identidad de secuencia (vs. secuencia original más corta): 37.96%

--- Comparando 1GP2:A con 1POA:A ---
  Puntuación de alineación: 66.10
  Identidad de secuencia (vs. secuencia original más corta): 54.24%

--- Comparando 1GP2:A con 1K6U:A ---
  Puntuación de alineación: 34.50
  Identidad de secuencia (vs. secuencia original más corta): 72.41%

--- Comparando 1GP2:A con 5DM2:A ---
  Puntuación de alineación: 123.20
  Identidad de secuencia (vs. secuencia original más corta): 40.38%

--- Interpretación --- 
Una mayor puntuación de alineación y un mayor porcentaje de identidad de secuencia indican una mayor similitud entre las proteínas. Las regiones con alta similitud pueden corresponder a dominios estructurales o funcionales conservados, incluyendo potencialmente los sitios activos. Sin e

## 1. Mapeo de "Epitopos Crípticos" mediante SASA (Solvent Accessible Surface Area)

Pregunta: ¿Existen regiones de alta accesibilidad en Der p 1 (5DM2) que no coincidan con los epítopos de unión a IgE conocidos (mAbs 5H8/10B9) y que puedan usarse para diseñar vacunas que no activen mastocitos?

In [24]:
import py3Dmol
from Bio.PDB import PDBList, PDBParser
from Bio.PDB.DSSP import DSSP
import os
import shutil

# Verificar e instalar DSSP si no está presente
if shutil.which("mkdssp") is None:
    print("mkdssp no encontrado. Intentando instalar...")
    !apt-get update
    !apt-get install -y dssp
    if shutil.which("mkdssp") is None:
        print("Error: mkdssp no pudo ser instalado. La accesibilidad a la superficie no se calculará.")
        dssp_available = False
    else:
        print("mkdssp instalado exitosamente.")
        dssp_available = True
else:
    print("mkdssp ya está instalado.")
    dssp_available = True

if dssp_available:
    pdb_id = '5DM2'
    print(f"\nProcesando Der p 1 (PDB ID: {pdb_id})...")

    # Descargar el archivo PDB
    pdb_list = PDBList()
    pdb_file = pdb_list.retrieve_pdb_file(pdb_id, pdir='.', file_format='pdb')

    # Parsear la estructura
    parser = PDBParser()
    structure = parser.get_structure(pdb_id, pdb_file)
    model = structure[0] # Asumiendo un solo modelo

    # Max accessibility values for standard amino acids (from Tien et al., 2013, J. Phys. Chem. B)
    max_acc_values = {
        'A': 113.0, 'R': 241.0, 'N': 158.0, 'D': 151.0, 'C': 140.0,
        'Q': 189.0, 'E': 183.0, 'G': 85.0, 'H': 194.0, 'I': 182.0,
        'L': 180.0, 'K': 213.0, 'M': 205.0, 'F': 218.0, 'P': 143.0,
        'S': 122.0, 'T': 146.0, 'W': 259.0, 'Y': 229.0, 'V': 160.0
    }
    RSA_THRESHOLD = 0.25 # Umbral para la accesibilidad a la superficie relativa (25%)

    surface_residues_to_highlight = []

    # Iterar sobre las cadenas y calcular DSSP
    for chain in model:
        print(f"  Calculando accesibilidad a la superficie para la cadena {chain.id}...")
        try:
            dssp = DSSP(model, pdb_file, dssp='mkdssp')
            for res_key in dssp.keys():
                # res_key es (chain_id, res_id_tuple), dssp_data es (residue_key, amino_acid, ss, phi, psi, asa, z_value, bp1, bp2, sheet_label, pure_dssp_code)
                current_chain_id, res_id_tuple = res_key
                if current_chain_id == chain.id: # Solo procesar residuos de la cadena actual
                    dssp_data = dssp[res_key]
                    one_letter_code = dssp_data[1] # Código de un caracter del aminoácido
                    asa = dssp_data[3] # Accesibilidad a la superficie absoluta

                    if one_letter_code in max_acc_values:
                        max_asa = max_acc_values[one_letter_code]
                        rsa = (asa / max_asa) if max_asa > 0 else 0

                        if rsa > RSA_THRESHOLD:
                            surface_residues_to_highlight.append({
                                'chain': chain.id,
                                'resi': res_id_tuple[1], # Número de residuo PDB
                                'rsa': rsa
                            })
        except Exception as e:
            print(f"    Advertencia: No se pudo ejecutar DSSP para la cadena {chain.id}: {e}")
            print("    Continuando sin calcular RSA para esta cadena.")

    print(f"\nSe encontraron {len(surface_residues_to_highlight)} residuos altamente accesibles (RSA > {RSA_THRESHOLD*100}%) en {pdb_id}.")

    # Visualización 3D con py3Dmol
    view = py3Dmol.view(query=f'pdb:{pdb_id}')
    view.setStyle({'cartoon': {'color': 'lightgray'}})

    # Resaltar residuos accesibles en un color distintivo (verde)
    for res_info in surface_residues_to_highlight:
        view.addStyle({
            'chain': res_info['chain'],
            'resi': res_info['resi']
        }, {'sphere': {'radius': 0.8, 'color': 'green'}})

    view.addSurface(py3Dmol.VDW, {'opacity':0.3, 'color': 'lightgray'})
    view.zoomTo()
    view.show()

    print("\n--- Interpretación de la Visualización ---")
    print("La estructura de Der p 1 (5DM2) se muestra en gris. Las esferas verdes resaltan los residuos predichos como altamente accesibles a la superficie (RSA > 25%). Estas son las regiones que podrían ser consideradas para el diseño de vacunas, una vez que se excluyan los epítopos de unión a IgE conocidos.")

    # Limpiar el archivo PDB descargado
    if os.path.exists(pdb_file):
        os.remove(pdb_file)
else:
    print("No se puede continuar sin DSSP. Por favor, asegúrese de que mkdssp esté disponible en el entorno.")

mkdssp no encontrado. Intentando instalar...
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [88.5 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,989 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,917 kB]
Get:13 https://ppa.launchpadcontent.net/d

3Dmol.js failed to load for some reason. Please check your browser console for error messages.


--- Interpretación de la Visualización ---
La estructura de Der p 1 (5DM2) se muestra en gris. Las esferas verdes resaltan los residuos predichos como altamente accesibles a la superficie (RSA > 25%). Estas son las regiones que podrían ser consideradas para el diseño de vacunas, una vez que se excluyan los epítopos de unión a IgE conocidos.


## 2.2. Diseño in silico de Hipoalérgenos (Alanine Scanning Virtual)
¿Qué mutaciones puntuales en el sitio activo de Der p 1 (Cys132, His170 en 5DM2) o en sus superficies de contacto eliminan su actividad proteolítica y su afinidad por IgE, manteniendo su estructura tridimensional para la inducción de IgG4?

In [25]:
# Para identificar mutaciones puntuales con los efectos deseados, se requerirían herramientas avanzadas de bioinformática y modelado molecular que no están disponibles en este entorno.
# Sin embargo, puedo ayudarte a visualizar los residuos clave mencionados (Cys132, His170) si me proporcionas la información del mapeo de epítopos de IgE y superficies de contacto.

## 3. Análisis de Distancia de Entrechuzamiento (Cross-linking) de IgEPregunta:
¿Cuál es la distancia euclidiana entre los centros de masa de los epítopos conocidos en 5DM2, y permite esta geometría el entrecruzamiento de dos moléculas de IgE en un receptor $Fc\epsilon RI$?

In [26]:
# Para calcular la distancia euclidiana entre los centros de masa de los epítopos,
# necesito que me proporciones los rangos de residuos específicos que definen cada 'epítopo conocido'.
# Por favor, introduce esta información en un formato que pueda procesar, como una lista de diccionarios:
# Ejemplo: known_epitopes_residues = [
#     {'name': 'Epitopo A', 'chain': 'A', 'start_resi': 10, 'end_resi': 20},
#     {'name': 'Epitopo B', 'chain': 'A', 'start_resi': 50, 'end_resi': 65},
# ]
# Una vez que tenga esta información, puedo calcular los centros de masa y las distancias.

## 4. Simulación de Interacción con Proteínas G y Señalización Innata

¿Existe mimetismo estructural o posibilidad de acoplamiento entre los residuos de contacto de Der p 1 y las subunidades de la Proteína G (1GP2) involucradas en la quimiotaxis de células Th2?



In [27]:
import py3Dmol

# PDB IDs para Der p 1 (5DM2) y la subunidad de Proteína G (1GP2)
pdb_id_derp1 = '5DM2'
pdb_id_gprotein = '1GP2'

print(f"Superponiendo Der p 1 ({pdb_id_derp1}) y Proteína G ({pdb_id_gprotein})...")

# Crear un visor y cargar la primera estructura (Der p 1)
view = py3Dmol.view(query=f'pdb:{pdb_id_derp1}')
view.setStyle({'cartoon': {'color': 'red'}})
view.addSurface(py3Dmol.VDW, {'opacity':0.3, 'color': 'lightcoral'})

# Cargar la segunda estructura (Proteína G) y superponerla
view.addModel(f"https://files.rcsb.org/download/{pdb_id_gprotein}.pdb", 'pdb')
view.setStyle({'model': 1}, {'cartoon': {'color': 'blue'}})
view.addSurface({'model': 1}, py3Dmol.VDW, {'opacity':0.3, 'color': 'lightblue'})

# Realizar la superposición
view.superposition(viewer=1, target=0)

view.zoomTo()
print("--- Visualización de Superposición de Der p 1 (rojo) y Proteína G (azul) ---")
view.show()

print("\n--- Interpretación --- ")
print(f"En esta visualización, Der p 1 ({pdb_id_derp1}) se muestra en rojo y la subunidad de Proteína G ({pdb_id_gprotein}) en azul. La superposición permite una comparación visual directa de sus formas tridimensionales. Las regiones donde las estructuras se solapan o muestran una gran similitud espacial podrían ser indicativas de mimetismo estructural. Sin embargo, para confirmar interacciones de acoplamiento, se requeriría un análisis más detallado como el docking molecular.")
print("Si tienes residuos de contacto específicos en mente para cualquiera de las dos proteínas, puedo resaltarlos en la visualización para un examen más detallado.")

Superponiendo Der p 1 (5DM2) y Proteína G (1GP2)...
--- Visualización de Superposición de Der p 1 (rojo) y Proteína G (azul) ---


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


--- Interpretación --- 
En esta visualización, Der p 1 (5DM2) se muestra en rojo y la subunidad de Proteína G (1GP2) en azul. La superposición permite una comparación visual directa de sus formas tridimensionales. Las regiones donde las estructuras se solapan o muestran una gran similitud espacial podrían ser indicativas de mimetismo estructural. Sin embargo, para confirmar interacciones de acoplamiento, se requeriría un análisis más detallado como el docking molecular.
Si tienes residuos de contacto específicos en mente para cualquiera de las dos proteínas, puedo resaltarlos en la visualización para un examen más detallado.
